<a href="https://colab.research.google.com/github/adriansivina-droid/scraper-mef/blob/main/EXPORT/EXP_Infotrade_Exportaciones.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ═══ BLOQUE 1 · INSTALACIÓN (ejecutar una vez por sesión) ═══
!pip install requests pandas numpy ipywidgets google-api-python-client --quiet
print("✅ Librerías listas")
import os, io, re, json, time, random, unicodedata, calendar, hashlib
import requests
import numpy as np
import pandas as pd
import ipywidgets as w
from datetime import date
from IPython.display import display

In [ ]:
# ═══ BLOQUE 2 · SELECTORES (elige y luego ejecuta el Bloque 3) ═══
# Regiones: selección múltiple (Ctrl/Cmd + clic). Sector y Subsector: una opción a la vez.
REGIONES = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho", "Cajamarca", "Callao",
            "Cusco", "Huancavelica", "Huánuco", "Ica", "Junín", "La Libertad", "Lambayeque",
            "Lima Metropolitana", "Loreto", "Madre de Dios", "Moquegua", "Pasco", "Piura",
            "Puno", "San Martín", "Tacna", "Tumbes", "Ucayali"]
SECTORES = ["Todos", "No Tradicional", "Tradicional", "Sin clasificación"]
SUBSECTORES = ["Todos", "AGROPECUARIO", "ARTESANÍAS", "MADERAS Y PAPELES", "METAL-MECÁNICO",
               "MINERÍA NO METÁLICA", "MINEROS", "PESQUERO", "PETRÓLEO Y GAS NATURAL",
               "PIELES Y CUEROS", "QUÍMICO", "SIDERO-METALÚRGICO", "SIN CLASIFICACIÓN",
               "TEXTIL", "VARIOS (inc. joyería)"]
CAPITULOS = {  # Sistema Armonizado (77 está reservado)
    "01": "Animales vivos",
    "02": "Carne y despojos comestibles",
    "03": "Pescados y crustáceos, moluscos y demás invertebrados acuáticos",
    "04": "Leche y productos lácteos; huevos de ave; miel natural; productos comestibles de origen animal",
    "05": "Los demás productos de origen animal",
    "06": "Plantas vivas y productos de la floricultura",
    "07": "Hortalizas, plantas, raíces y tubérculos alimenticios",
    "08": "Frutas y frutos comestibles; cortezas de agrios (cítricos), melones o sandías",
    "09": "Café, té, yerba mate y especias",
    "10": "Cereales",
    "11": "Productos de la molinería; malta; almidón y fécula; inulina; gluten de trigo",
    "12": "Semillas y frutos oleaginosos; semillas y frutos diversos; plantas industriales o medicinales; paja y forraje",
    "13": "Gomas, resinas y demás jugos y extractos vegetales",
    "14": "Materias trenzables y demás productos de origen vegetal",
    "15": "Grasas y aceites animales, vegetales o de origen microbiano; ceras",
    "16": "Preparaciones de carne, pescado, crustáceos, moluscos o insectos",
    "17": "Azúcares y artículos de confitería",
    "18": "Cacao y sus preparaciones",
    "19": "Preparaciones a base de cereales, harina, almidón, fécula o leche; productos de pastelería",
    "20": "Preparaciones de hortalizas, frutas u otros frutos o demás partes de plantas",
    "21": "Preparaciones alimenticias diversas",
    "22": "Bebidas, líquidos alcohólicos y vinagre",
    "23": "Residuos y desperdicios de las industrias alimentarias; alimentos para animales",
    "24": "Tabaco y sucedáneos del tabaco elaborados",
    "25": "Sal; azufre; tierras y piedras; yesos, cales y cementos",
    "26": "Minerales metalíferos, escorias y cenizas",
    "27": "Combustibles minerales, aceites minerales y productos de su destilación; ceras minerales",
    "28": "Productos químicos inorgánicos; compuestos de metal precioso, tierras raras o elementos radiactivos",
    "29": "Productos químicos orgánicos",
    "30": "Productos farmacéuticos",
    "31": "Abonos",
    "32": "Extractos curtientes o tintóreos; pigmentos, pinturas y barnices; tintas",
    "33": "Aceites esenciales y resinoides; preparaciones de perfumería, de tocador o de cosmética",
    "34": "Jabón, agentes de superficie orgánicos, preparaciones para lavar, lubricantes, ceras, velas",
    "35": "Materias albuminoideas; almidones modificados; colas; enzimas",
    "36": "Pólvora y explosivos; artículos de pirotecnia; fósforos; materias inflamables",
    "37": "Productos fotográficos o cinematográficos",
    "38": "Productos diversos de las industrias químicas",
    "39": "Plástico y sus manufacturas",
    "40": "Caucho y sus manufacturas",
    "41": "Pieles (excepto la peletería) y cueros",
    "42": "Manufacturas de cuero; artículos de talabartería; artículos de viaje, bolsos",
    "43": "Peletería y confecciones de peletería; peletería facticia o artificial",
    "44": "Madera, carbón vegetal y manufacturas de madera",
    "45": "Corcho y sus manufacturas",
    "46": "Manufacturas de espartería o cestería",
    "47": "Pasta de madera o de las demás materias fibrosas celulósicas; papel o cartón para reciclar",
    "48": "Papel y cartón; manufacturas de pasta de celulosa, de papel o cartón",
    "49": "Productos editoriales, de la prensa y de otras industrias gráficas",
    "50": "Seda",
    "51": "Lana y pelo fino u ordinario; hilados y tejidos de crin",
    "52": "Algodón",
    "53": "Las demás fibras textiles vegetales; hilados de papel",
    "54": "Filamentos sintéticos o artificiales",
    "55": "Fibras sintéticas o artificiales discontinuas",
    "56": "Guata, fieltro y tela sin tejer; hilados especiales; cordeles, cuerdas",
    "57": "Alfombras y demás revestimientos para el suelo, de materia textil",
    "58": "Tejidos especiales; superficies textiles con mechón insertado; encajes; tapicería",
    "59": "Telas impregnadas, recubiertas, revestidas o estratificadas; artículos técnicos de materia textil",
    "60": "Tejidos de punto",
    "61": "Prendas y complementos de vestir, de punto",
    "62": "Prendas y complementos de vestir, excepto los de punto",
    "63": "Los demás artículos textiles confeccionados; prendería y trapos",
    "64": "Calzado, polainas y artículos análogos; partes",
    "65": "Sombreros, demás tocados, y sus partes",
    "66": "Paraguas, sombrillas, bastones, látigos, fustas",
    "67": "Plumas y plumón preparados; flores artificiales; manufacturas de cabello",
    "68": "Manufacturas de piedra, yeso, cemento, amianto, mica o materias análogas",
    "69": "Productos cerámicos",
    "70": "Vidrio y sus manufacturas",
    "71": "Perlas, piedras preciosas o semipreciosas, metales preciosos y sus manufacturas; bisutería; monedas",
    "72": "Fundición, hierro y acero",
    "73": "Manufacturas de fundición, hierro o acero",
    "74": "Cobre y sus manufacturas",
    "75": "Níquel y sus manufacturas",
    "76": "Aluminio y sus manufacturas",
    "78": "Plomo y sus manufacturas",
    "79": "Cinc y sus manufacturas",
    "80": "Estaño y sus manufacturas",
    "81": "Los demás metales comunes; cermets; manufacturas de estas materias",
    "82": "Herramientas y útiles, artículos de cuchillería y cubiertos, de metal común",
    "83": "Manufacturas diversas de metal común",
    "84": "Reactores nucleares, calderas, máquinas, aparatos y artefactos mecánicos; partes",
    "85": "Máquinas, aparatos y material eléctrico, y sus partes; aparatos de grabación o reproducción de sonido e imagen",
    "86": "Vehículos y material para vías férreas; aparatos de señalización",
    "87": "Vehículos automóviles, tractores, velocípedos y demás vehículos terrestres; partes",
    "88": "Aeronaves, vehículos espaciales, y sus partes",
    "89": "Barcos y demás artefactos flotantes",
    "90": "Instrumentos y aparatos de óptica, fotografía, cinematografía, medida, control, precisión o médico-quirúrgicos",
    "91": "Aparatos de relojería y sus partes",
    "92": "Instrumentos musicales; sus partes y accesorios",
    "93": "Armas, municiones, y sus partes y accesorios",
    "94": "Muebles; mobiliario médico-quirúrgico; artículos de cama; luminarias; construcciones prefabricadas",
    "95": "Juguetes, juegos y artículos para recreo o deporte; sus partes",
    "96": "Manufacturas diversas",
    "97": "Objetos de arte o colección y antigüedades",
    "98": "Mercancías con tratamiento especial (capítulo nacional)"}
MESES = ["Enero", "Febrero", "Marzo", "Abril", "Mayo", "Junio", "Julio", "Agosto",
         "Setiembre", "Octubre", "Noviembre", "Diciembre"]

_hoy = date.today()
_disp = (_hoy.year * 12 + _hoy.month - 1) - 3          # por defecto: hoy menos 3 meses
_años = list(range(_hoy.year, 1993, -1))                 # la web ofrece 1994 en adelante
_mes = [(m, i + 1) for i, m in enumerate(MESES)]
_ancho = {"description_width": "150px"}

sel_anio_ini = w.Dropdown(options=_años, value=2005, description="Año inicio", style=_ancho)
sel_mes_ini = w.Dropdown(options=_mes, value=1, description="Mes inicio", style=_ancho)
sel_anio_disp = w.Dropdown(options=_años, value=_disp // 12, description="Disponible hasta (año)", style=_ancho)
sel_mes_disp = w.Dropdown(options=_mes, value=_disp % 12 + 1, description="Disponible hasta (mes)", style=_ancho)
sel_regiones = w.SelectMultiple(options=REGIONES, value=tuple(REGIONES), rows=13,
                                description="Regiones", style=_ancho,
                                layout=w.Layout(width="420px"))
btn_todas = w.Button(description="Marcar todas", layout=w.Layout(width="130px"))
btn_ninguna = w.Button(description="Desmarcar todas", layout=w.Layout(width="130px"))
btn_todas.on_click(lambda _: setattr(sel_regiones, "value", tuple(REGIONES)))
btn_ninguna.on_click(lambda _: setattr(sel_regiones, "value", ()))
sel_sector = w.Dropdown(options=SECTORES, value="Todos", description="Sector", style=_ancho)
sel_subsector = w.Dropdown(options=SUBSECTORES, value="Todos", description="Subsector", style=_ancho,
                           layout=w.Layout(width="420px"))
sel_capitulos = w.SelectMultiple(options=[(f"{k} · {v}"[:95], k) for k, v in CAPITULOS.items()],
                                 value=(), rows=10, description="Capítulos", style=_ancho,
                                 layout=w.Layout(width="760px"))
btn_sin_cap = w.Button(description="Quitar capítulos", layout=w.Layout(width="130px"))
btn_sin_cap.on_click(lambda _: setattr(sel_capitulos, "value", ()))
# Lista incluida en el notebook (se usa si la web no entrega la suya). Los nombres se comparan sin
# mayúsculas ni tildes con la columna «Mercado destino»; si alguno no coincide, el Bloque 3 avisa.
PAISES_BASE = [p.strip() for p in """Afganistán, Albania, Alemania, Andorra, Angola, Antigua y Barbuda, Antillas Neerlandesas, Arabia Saudita, Argelia,
Argentina, Armenia, Aruba, Australia, Austria, Azerbaiyán, Bahamas, Bahréin, Bangladesh, Barbados, Bélgica, Belice,
Benín, Bermudas, Bielorrusia, Bolivia, Bosnia y Herzegovina, Botsuana, Brasil, Brunéi, Bulgaria, Burkina Faso,
Burundi, Bután, Cabo Verde, Camboya, Camerún, Canadá, Catar, Chad, Chile, China, Chipre, Colombia, Comoras, Congo,
Corea del Norte, Corea del Sur, Costa de Marfil, Costa Rica, Croacia, Cuba, Curazao, Dinamarca, Dominica, Ecuador,
Egipto, El Salvador, Emiratos Árabes Unidos, Eritrea, Eslovaquia, Eslovenia, España, Estados Unidos, Estonia, Etiopía,
Filipinas, Finlandia, Fiyi, Francia, Gabón, Gambia, Georgia, Ghana, Gibraltar, Granada, Grecia, Groenlandia, Guadalupe,
Guam, Guatemala, Guayana Francesa, Guinea, Guinea Ecuatorial, Guinea-Bisáu, Guyana, Haití, Honduras, Hong Kong,
Hungría, India, Indonesia, Irak, Irán, Irlanda, Islandia, Islas Caimán, Islas Marshall, Islas Salomón, Islas Vírgenes Británicas,
Israel, Italia, Jamaica, Japón, Jordania, Kazajistán, Kenia, Kirguistán, Kiribati, Kosovo, Kuwait, Laos, Lesoto, Letonia,
Líbano, Liberia, Libia, Liechtenstein, Lituania, Luxemburgo, Macao, Macedonia del Norte, Madagascar, Malasia, Malaui,
Maldivas, Malí, Malta, Marruecos, Martinica, Mauricio, Mauritania, México, Micronesia, Moldavia, Mónaco, Mongolia,
Montenegro, Mozambique, Myanmar, Namibia, Nepal, Nicaragua, Níger, Nigeria, Noruega, Nueva Caledonia, Nueva Zelanda,
Omán, Países Bajos, Pakistán, Palaos, Palestina, Panamá, Papúa Nueva Guinea, Paraguay, Polinesia Francesa, Polonia,
Portugal, Puerto Rico, Reino Unido, República Centroafricana, República Checa, República Democrática del Congo,
República Dominicana, Reunión, Ruanda, Rumania, Rusia, Samoa, San Cristóbal y Nieves, San Marino, San Vicente y las Granadinas,
Santa Lucía, Santo Tomé y Príncipe, Senegal, Serbia, Seychelles, Sierra Leona, Singapur, Siria, Somalia, Sri Lanka,
Sudáfrica, Sudán, Sudán del Sur, Suecia, Suiza, Surinam, Tailandia, Taiwán, Tanzania, Tayikistán, Timor Oriental, Togo,
Tonga, Trinidad y Tobago, Túnez, Turkmenistán, Turquía, Tuvalu, Ucrania, Uganda, Uruguay, Uzbekistán, Vanuatu, Venezuela,
Vietnam, Yemen, Yibuti, Zambia, Zimbabue""".replace("\n", " ").split(",")]

def cargar_paises():
    """Lista de países de la web (la de su desplegable «Mercado»). -> (lista, motivo si falló)."""
    base = "https://infotrade.promperu.gob.pe"
    cab = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/124.0 Safari/537.36", "Accept": "application/json",
           "Origin": base, "Referer": f"{base}/reporte-exportaciones"}
    s = requests.Session()
    s.headers.update(cab)
    try:
        s.get(f"{base}/reporte-exportaciones", timeout=20)       # cookies del sitio, como el navegador
    except Exception:
        pass
    motivo = "sin respuesta"
    def listas(x):                               # todas las listas de diccionarios dentro de la respuesta
        if isinstance(x, list) and x and all(isinstance(i, dict) for i in x):
            yield x
        for v in (x.values() if isinstance(x, dict) else x if isinstance(x, list) else []):
            yield from listas(v)
    intentos = [(m, f"{base}/wss/api/{ruta}") for ruta in
                ("Comunes?TIPOCONSULTA=EjecutarPais", "Comunes/Comunes?TIPOCONSULTA=EjecutarPais")
                for m in ("GET", "POST")]
    for metodo, url in intentos:
        try:
            r = s.request(metodo, url, json={} if metodo == "POST" else None, timeout=20)
            motivo = f"HTTP {r.status_code}"
            datos = r.json()
        except Exception as e:
            motivo = motivo if "HTTP" in motivo else type(e).__name__
            continue
        mejor = []
        for lst in listas(datos):
            for k in lst[0]:
                vals = {str(d.get(k, "")).strip() for d in lst}
                vals = {v for v in vals if re.search(r"[A-Za-zÁÉÍÓÚáéíóúÑñ]{3}", v)
                        and v.lower() not in ("todos", "seleccione", "--seleccione--")}
                prefer = any(t in k.lower() for t in ("nombre", "descrip", "pais"))
                if (len(vals), prefer) > (len(mejor), False):
                    mejor = sorted(vals)
        if len(mejor) >= 20:
            return mejor, ""
    return [], motivo

PAISES_WEB, _motivo = cargar_paises()
PAISES = PAISES_WEB or PAISES_BASE
print(f"🌎 Países: lista de la web ({len(PAISES)})" if PAISES_WEB else
      f"🌎 Países: se usa la lista incluida ({len(PAISES)}); la web no la entregó ({_motivo}).")
sel_paises = w.SelectMultiple(options=PAISES, value=(), rows=10, description="Países destino",
                              style=_ancho, layout=w.Layout(width="420px"))
btn_sin_pais = w.Button(description="Quitar países", layout=w.Layout(width="130px"))
btn_sin_pais.on_click(lambda _: setattr(sel_paises, "value", ()))
txt_paises = w.Text(value="", placeholder="otros, separados por coma (p. ej. China, Estados Unidos)",
                    description="Países (texto)", style=_ancho, layout=w.Layout(width="560px"))
txt_partida = w.Text(value="", placeholder="vacío = todas (p. ej. 4418990000)", description="Partida", style=_ancho)
txt_empresa = w.Text(value="", placeholder="vacío = todas", description="Empresa", style=_ancho)
txt_usuario = w.Text(value="", placeholder="opcional: tu usuario de Infotrade", description="Usuario (auditoría)", style=_ancho)

display(w.VBox([
    w.HTML("<b>Período</b> (se descarga en cortes de 6 meses: Ene–Jun y Jul–Dic). "
           "En «Disponible hasta» pon el mes que indica la web: «Datos disponibles hasta …»."),
    w.HBox([sel_anio_ini, sel_mes_ini]),
    w.HBox([sel_anio_disp, sel_mes_disp]),
    w.HTML("<b>Regiones</b> (Ctrl/Cmd + clic para elegir varias)"),
    w.HBox([sel_regiones, w.VBox([btn_todas, btn_ninguna])]),
    w.HTML("<b>Filtros</b> (Sector y Subsector: una opción a la vez)"),
    sel_sector, sel_subsector,
    w.HTML("<b>Capítulos</b> (opcional, Ctrl/Cmd + clic para varios). La web solo acepta partidas de "
           "10 dígitos, así que se descarga todo el corte y se guardan solo las filas de estos capítulos. "
           "Sin capítulos = todas las partidas. No combinar con «Partida»."),
    w.HBox([sel_capitulos, btn_sin_cap]),
    w.HTML("<b>País de destino</b> (opcional). Se descarga todo el corte y se guardan solo las filas de "
           "estos países. Sin países = todos (deja el campo de texto vacío). Si un país no está en la lista, "
           "escríbelo abajo tal como aparece en la columna «Mercado destino»."),
    w.HBox([sel_paises, btn_sin_pais]), txt_paises,
    txt_partida, txt_empresa, txt_usuario,
]))

In [ ]:
# ═══ BLOQUE 3 · DESCARGA, CONTROL Y DRIVE ═══
# Usa lo elegido en el Bloque 2. Si se corta (Colab se desconecta), vuelve a ejecutarlo:
# salta los cortes que ya están en Drive con control OK y sigue donde quedó.

# ── 1. CONFIGURACIÓN FIJA ────────────────────────────────────────────────────
DRIVE_FOLDER_ID = "1JJqcvUomccL-dCCNm5yovl-9TN5laKYi"           # CSV de datos
DRIVE_FOLDER_CONTROL_ID = "1gAO8-UjTV2bmBwMPgONBAYeVfpwXSWCY"   # CONTROL y LOG
PAUSA = (2.0, 4.0)             # segundos aleatorios entre cortes (cortesía con el servidor)
TIMEOUT_EXCEL = 600            # segundos: la descarga de un corte grande puede tardar

API = "https://infotrade.promperu.gob.pe/wss/api"
PAGINA = "https://infotrade.promperu.gob.pe/reporte-exportaciones"
ruta_guardado = "/content/Exportaciones_Infotrade"
os.makedirs(ruta_guardado, exist_ok=True)

# ── 2. LEER LOS SELECTORES ───────────────────────────────────────────────────
inicio = (sel_anio_ini.value, sel_mes_ini.value)
fin = (sel_anio_disp.value, sel_mes_disp.value)
regiones = list(sel_regiones.value)
SECTOR, SUBSECTOR = sel_sector.value, sel_subsector.value
PARTIDA, EMPRESA = txt_partida.value.strip(), txt_empresa.value.strip()
USUARIO = txt_usuario.value.strip()
CAPS = sorted(sel_capitulos.value)
if PARTIDA and CAPS:
    raise ValueError("Elige una Partida o Capítulos, no ambos (en el Bloque 2 pulsa «Quitar capítulos» o borra la partida).")
if len(CAPS) == len(CAPITULOS):
    CAPS = []                                     # todos los capítulos = sin filtro
def norm_txt(t):
    t = unicodedata.normalize("NFKD", str(t or ""))
    return " ".join("".join(c for c in t if not unicodedata.combining(c)).lower().split())
_paises = {norm_txt(p): p.strip() for p in list(sel_paises.value) + txt_paises.value.split(",")
           if p.strip() and norm_txt(p) not in ("todos", "todas", "todo")}       # «todos» = sin filtro
PAISES_SEL = sorted(_paises.values(), key=norm_txt)        # países de destino elegidos ([] = todos)
PAISES_N = set(_paises)
# Prefijo de los archivos: partida, capítulos o TODAS (+ países)
if PARTIDA:
    PREFIJO = PARTIDA
elif CAPS:
    PREFIJO = "CAP" + "+".join(CAPS) if len(CAPS) <= 6 else f"CAP{len(CAPS)}caps-{hashlib.md5('+'.join(CAPS).encode()).hexdigest()[:6]}"
else:
    PREFIJO = "TODAS"
ETIQ_CAP = PREFIJO[3:] if CAPS else ""
if PAISES_SEL:
    ETIQ_PAIS = ("+".join(PAISES_SEL) if len(PAISES_SEL) <= 3 else
                 f"{len(PAISES_SEL)}paises-{hashlib.md5('+'.join(sorted(PAISES_N)).encode()).hexdigest()[:6]}")
    ETIQ_PAIS = re.sub(r'[\\/:*?"<>|]', "-", ETIQ_PAIS)
    PREFIJO += f" [{ETIQ_PAIS}]"
if inicio > fin:
    raise ValueError(f"El inicio {inicio} es posterior a «Disponible hasta» {fin}.")
if not regiones:
    raise ValueError("Elige al menos una región en el Bloque 2.")

def cortes(ini, fin):
    """Semestres (Ene–Jun / Jul–Dic) recortados al rango [ini, fin]. -> (año, sem, (a,m), (a,m), completo)"""
    out = []
    for a in range(ini[0], fin[0] + 1):
        for s, (m1, m2) in ((1, (1, 6)), (2, (7, 12))):
            d, h = max((a, m1), ini), min((a, m2), fin)
            if d <= h:
                out.append((a, s, d, h, (d, h) == ((a, m1), (a, m2))))
    return out

CORTES = cortes(inicio, fin)
ETIQ_FILTRO = " - ".join(x for x in [
    f"Sector {SECTOR}", f"Subsector {SUBSECTOR}",
    f"Partida {PARTIDA}" if PARTIDA else "", f"Capitulos {ETIQ_CAP}" if CAPS else "",
    f"Mercados {ETIQ_PAIS}" if PAISES_SEL else "", f"Empresa {EMPRESA}" if EMPRESA else ""] if x)
print(f"Período: {MESES[inicio[1]-1]} {inicio[0]} – {MESES[fin[1]-1]} {fin[0]}  →  {len(CORTES)} corte(s)")
print(f"Regiones: {len(regiones)}  |  {ETIQ_FILTRO}")
if CAPS:
    print(f"Capítulos: {', '.join(CAPS)}  (se descarga el corte completo y se guardan solo esas filas)")
if PAISES_SEL:
    print(f"Países de destino: {', '.join(PAISES_SEL)}  (se descarga el corte completo y se guardan solo esas filas)")
print(f"Descargas posibles: {len(CORTES) * len(regiones)}\n")

def nombre_dato(d, h, region):
    """'[partida | CAPxx | TODAS] - AAAAMM-AAAAMM - [región].csv'."""
    return f"{PREFIJO} - {d[0]}{d[1]:02d}-{h[0]}{h[1]:02d} - {region}.csv"

def versiones_viejas(año, sem, region, nombre_nuevo):
    """Archivos del mismo semestre y región con otro rango de meses (p. ej. un parcial
    202607-202607 que queda reemplazado por 202607-202608)."""
    patron = re.compile(rf"^{re.escape(PREFIJO)} - (\d{{4}})(\d{{2}})-\d{{6}} - {re.escape(region)}\.csv$")
    viejos = []
    for n in datos_drive:
        m = patron.match(n)
        if m and n != nombre_nuevo and int(m.group(1)) == año and (int(m.group(2)) <= 6) == (sem == 1):
            viejos.append(n)
    return viejos

NOMBRE_CONTROL = f"CONTROL Exportaciones - {ETIQ_FILTRO}.csv"

# ── 3. GOOGLE DRIVE ──────────────────────────────────────────────────────────
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
auth.authenticate_user()
drive = build("drive", "v3", cache_discovery=False)

for fid in (DRIVE_FOLDER_ID, DRIVE_FOLDER_CONTROL_ID):
    try:
        c = drive.files().get(fileId=fid, fields="id,name,mimeType,capabilities(canAddChildren)",
                              supportsAllDrives=True).execute()
    except HttpError as e:
        raise PermissionError(f"Sin acceso a la carpeta de Drive {fid} (HTTP {e.resp.status}).") from e
    if c.get("mimeType") != "application/vnd.google-apps.folder":
        raise ValueError(f"El ID {fid} no es una carpeta de Drive.")
    if not c.get("capabilities", {}).get("canAddChildren", False):
        raise PermissionError(f"Sin permiso de Editor en '{c['name']}'.")
    print(f"☁️  Drive conectado -> carpeta: '{c['name']}'")

def archivos_en(folder_id):
    """{nombre: id} de los archivos de una carpeta (con paginación)."""
    out, token = {}, None
    while True:
        r = drive.files().list(q=f"'{folder_id}' in parents and trashed = false",
                               fields="nextPageToken, files(id,name)", pageSize=1000, pageToken=token,
                               supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        out.update({f["name"]: f["id"] for f in r.get("files", [])})
        token = r.get("nextPageToken")
        if not token:
            return out

def subir(ruta_local, folder_id, existentes, reintentos=5):
    nombre = os.path.basename(ruta_local)
    for i in range(1, reintentos + 1):
        try:
            media = MediaFileUpload(ruta_local, mimetype="text/csv", resumable=True)
            if nombre in existentes:
                drive.files().update(fileId=existentes[nombre], media_body=media,
                                     supportsAllDrives=True).execute()
            else:
                f = drive.files().create(body={"name": nombre, "parents": [folder_id]}, media_body=media,
                                         fields="id", supportsAllDrives=True).execute()
                existentes[nombre] = f["id"]
            return True
        except Exception as e:
            print(f"   ↻ Drive reintento {i}/{reintentos}: {e}")
            time.sleep(2 ** i)
    print(f"   ❌ No se pudo subir {nombre}. Queda en {ruta_local}")
    return False

def leer_csv_drive(file_id):
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=file_id, supportsAllDrives=True))
    hecho = False
    while not hecho:
        _, hecho = dl.next_chunk()
    buf.seek(0)
    return pd.read_csv(buf, dtype=str, encoding="utf-8-sig")

datos_drive = archivos_en(DRIVE_FOLDER_ID)
control_drive = archivos_en(DRIVE_FOLDER_CONTROL_ID)

# Control acumulado de corridas anteriores (sirve para reanudar)
COLS_CONTROL = ["AÑO", "SEMESTRE", "DESDE", "HASTA", "CORTE_COMPLETO", "REGION", "SECTOR", "SUBSECTOR",
                "PARTIDA", "CAPITULOS", "MERCADOS", "EMPRESA", "FILAS_DESCARGADAS", "FILAS_REF", "DIF_FILAS",
                "FOB_USD_DESCARGADO", "FOB_USD_REF", "DIF_FOB_USD",
                "PESO_NETO_KG_DESCARGADO", "PESO_NETO_KG_REF", "DIF_PESO_NETO_KG",
                "PESO_BRUTO_KG_DESCARGADO", "PESO_BRUTO_KG_REF", "DIF_PESO_BRUTO_KG",
                "CANTIDAD_DESCARGADA", "CANTIDAD_REF", "DIF_CANTIDAD",
                "REGIONES_EN_DATOS", "FECHAS_INVALIDAS", "FILAS_GUARDADAS", "FOB_USD_GUARDADO", "ESTADO", "ARCHIVO", "FECHA_EXTRACCION"]
if NOMBRE_CONTROL in control_drive:
    control = leer_csv_drive(control_drive[NOMBRE_CONTROL])
    for c in COLS_CONTROL:                     # controles creados con una versión anterior del notebook
        if c not in control.columns:
            control[c] = ""
    print(f"📄 Control previo encontrado: {len(control)} corte(s) registrados")
else:
    control = pd.DataFrame(columns=COLS_CONTROL)

# ── 4. API DE INFOTRADE ──────────────────────────────────────────────────────
sesion = requests.Session()
sesion.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
    "Accept": "application/json", "Content-Type": "application/json",
    "Origin": "https://infotrade.promperu.gob.pe", "Referer": PAGINA})

def llamar(ruta, cuerpo, timeout=60, reintentos=4):
    """POST JSON con reintentos y backoff. -> (json, error)."""
    ultimo = ""
    for i in range(reintentos):
        try:
            r = sesion.post(f"{API}/{ruta}", json=cuerpo, timeout=timeout)
            if r.status_code == 200:
                return r.json(), None
            ultimo = f"HTTP {r.status_code}"
        except (requests.RequestException, ValueError) as e:
            ultimo = f"{type(e).__name__}: {e}"
        time.sleep(min(60, 3 * 2 ** i + random.random()))
    return None, f"{ruta}: {ultimo}"

def filtros(d, h, region):
    ult = calendar.monthrange(*h)[1]
    return {"FechaInicio": f"01-{d[1]:02d}-{d[0]}", "FechaFin": f"{ult:02d}-{h[1]:02d}-{h[0]}",
            "Empresa": EMPRESA, "Mercado": "", "Partida": PARTIDA, "Departamento": region,
            "Tipo": "" if SECTOR == "Todos" else SECTOR,
            "Sector": "" if SUBSECTOR == "Todos" else SUBSECTOR,
            "DescripcionComercial": "", "DescripcionProducto1": "", "Condicion2": "",
            "DescripcionProducto2": "", "Condicion3": "", "DescripcionProducto3": "",
            "DescripcionProducto1E": "", "Condicion2E": "", "DescripcionProducto2E": "",
            "Condicion3E": "", "DescripcionProducto3E": ""}

def auditoria(f, evento):
    if not USUARIO:
        return
    j = dict(f, Boton=evento, Navegador="Chrome", Dispositivo="Windows", Ip="")
    llamar("user/RegistrarAuditoria", {
        "UsuarioAuditoria": USUARIO, "CodigoOpcionAuditoria": "Reportes",
        "CodigoPaginaAuditoria": "Exportaciones", "CodigoSubPaginaAuditoria": "",
        "Evento": evento, "Json": json.dumps(j, ensure_ascii=False),
        "Pais": "Todos", "Partida": f["Partida"], "Ruc": ""}, reintentos=1)

COLUMNAS = {  # API -> CSV (nombres como en la web)
    "FechaEmbarque": "FECHA_EMBARQUE", "Ruc": "RUC", "RazonSocial": "RAZON_SOCIAL",
    "Departamento": "REGION", "CodigoPartida": "PARTIDA", "Partida": "DESCRIPCION_ARANCELARIA",
    "DescripcionComercial": "DESCRIPCION_COMERCIAL", "PaisDestino": "MERCADO_DESTINO",
    "Monto": "MONTO_FOB_USD", "PesoNeto": "PESO_NETO_KG", "PesoBruto": "PESO_BRUTO_KG",
    "Cantidad": "CANTIDAD", "TipoSector": "SECTOR", "Sector": "SUBSECTOR"}
NUMERICAS = {"Monto": "FOB_USD", "PesoNeto": "PESO_NETO_KG", "PesoBruto": "PESO_BRUTO_KG", "Cantidad": "CANTIDAD"}

def a_num(x):
    try:
        return float(str(x).replace(",", "")) if x not in (None, "") else 0.0
    except ValueError:
        return np.nan

def normalizar(t):
    t = unicodedata.normalize("NFKD", str(t or ""))
    return "".join(c for c in t if not unicodedata.combining(c)).lower().strip()

def descargar_corte(año, sem, d, h, completo, region):
    """Secuencia de la web: concurrencia -> consulta -> Excel -> concurrencia (1, 2) -> auditoría."""
    f = filtros(d, h, region)
    llamar("Comunes/ObtenerParametroConcurrenciaReporteActual", {"headers": {"Accept": "application/json"}})
    ref, err = llamar("Reportes/Exportaciones", dict(f, NumeroPagina=1))
    if err:
        return None, None, err
    total = int(ref.get("TotalRegistros") or 0)
    tot = ref.get("Totales") or {}
    filas = []
    if total > 0:
        exc, err = llamar("Reportes/ExportacionesExcel", f, timeout=TIMEOUT_EXCEL, reintentos=3)
        if err:
            return None, None, err
        filas = exc.get("RptData") or []
        llamar("Comunes/ActualizarConcurrenciaDescarga", {"Tipo": 1, "PerfilUsuario": "PERFIL_USUARIO_EXTERNO"}, reintentos=2)
        llamar("Comunes/ActualizarConcurrenciaDescarga", {"Tipo": 2, "PerfilUsuario": "PERFIL_USUARIO_EXTERNO"}, reintentos=2)
        auditoria(f, "EXPORTAR EXCEL")
    df = pd.DataFrame(filas)
    for c in COLUMNAS:
        if c not in df.columns:
            df[c] = ""
    df = df[list(COLUMNAS)]
    # ExportacionesExcel agrega al final una fila de TOTAL (sin fecha, RUC ni empresa):
    # se descarta para no duplicar montos ni contar una fila de más.
    vacio = lambda col: df[col].fillna("").astype(str).str.strip().isin(["", "None", "nan"])
    es_total = vacio("FechaEmbarque") & vacio("Ruc") & vacio("RazonSocial") & vacio("CodigoPartida")
    df = df[~es_total].reset_index(drop=True)
    for c in NUMERICAS:
        df[c] = pd.to_numeric(df[c].map(a_num), errors="coerce").fillna(0.0)
    for c in ("Ruc", "CodigoPartida"):          # texto: conservar ceros a la izquierda
        df[c] = df[c].astype(str).replace({"None": "", "nan": ""})
    # partida siempre en 10 dígitos (si llegara como número, recupera el 0 inicial: 0901...)
    df["CodigoPartida"] = df["CodigoPartida"].where(~df["CodigoPartida"].str.fullmatch(r"\d{7,9}"),
                                                    df["CodigoPartida"].str.zfill(10))
    # Fecha de embarque uniforme dd/mm/aaaa (con ceros) + AÑO y MES para el BI
    crudo = df["FechaEmbarque"].fillna("").astype(str).str.strip()
    fecha = pd.to_datetime(crudo, format="%d/%m/%Y", errors="coerce")
    otras = fecha.isna() & crudo.ne("")
    if otras.any():                             # otros formatos (p. ej. 2025-01-02T00:00:00)
        try:
            fecha[otras] = pd.to_datetime(crudo[otras], format="mixed", dayfirst=True, errors="coerce")
        except (TypeError, ValueError):
            fecha[otras] = pd.to_datetime(crudo[otras], dayfirst=True, errors="coerce")
    fechas_invalidas = int(fecha.isna().sum())
    df["FechaEmbarque"] = fecha.dt.strftime("%d/%m/%Y").where(fecha.notna(), crudo)
    df.insert(1, "AÑO", fecha.dt.year.astype("Int64"))
    df.insert(2, "MES", fecha.dt.month.astype("Int64"))

    # Control del corte: filas y totales vs la consulta de la web
    fila = {"AÑO": año, "SEMESTRE": f"S{sem}", "DESDE": f["FechaInicio"].replace("-", "/"),
            "HASTA": f["FechaFin"].replace("-", "/"),
            "CORTE_COMPLETO": "SI" if completo else "NO", "REGION": region, "SECTOR": SECTOR,
            "SUBSECTOR": SUBSECTOR, "PARTIDA": PARTIDA, "CAPITULOS": " ".join(CAPS),
            "MERCADOS": "; ".join(PAISES_SEL), "EMPRESA": EMPRESA,
            "FILAS_DESCARGADAS": len(df), "FILAS_REF": total, "DIF_FILAS": len(df) - total}
    estado = "OK" if len(df) == total else "REVISAR"
    for api, col in NUMERICAS.items():
        desc = float(df[api].sum()) if len(df) else 0.0
        refv = a_num(tot.get(api)) if total else 0.0
        suf = "DESCARGADA" if col == "CANTIDAD" else "DESCARGADO"
        fila[f"{col}_{suf}"], fila[f"{col}_REF"] = round(desc, 3), refv
        fila[f"DIF_{col}"] = round(desc - refv, 3) + 0.0          # + 0.0: evita "-0.0" en el control
        if abs(desc - refv) > max(1.0, abs(refv) * 1e-9):
            estado = "REVISAR"
    regiones_datos = sorted(set(df["Departamento"].astype(str)))
    fila["REGIONES_EN_DATOS"] = "; ".join(regiones_datos)
    pedida = normalizar(region)
    if any(not (normalizar(r) in pedida or pedida in normalizar(r)) for r in regiones_datos):
        estado = "REVISAR"
    # Filtro por capítulo: el control de arriba compara el corte completo con la web;
    # aquí se guardan solo las filas de los capítulos elegidos.
    if CAPS:
        cap = df["CodigoPartida"].str[:2]
        df = df[cap.isin(CAPS)].reset_index(drop=True)
    if PAISES_SEL:
        destino = df["PaisDestino"].map(norm_txt)
        paises_vistos.update(df["PaisDestino"].astype(str))
        encontrados.update(set(destino) & PAISES_N)
        df = df[destino.isin(PAISES_N)].reset_index(drop=True)
    fila["FILAS_GUARDADAS"] = len(df)
    fila["FOB_USD_GUARDADO"] = round(float(df["Monto"].sum()), 3) if len(df) else 0.0
    fila["FECHAS_INVALIDAS"] = fechas_invalidas
    if fechas_invalidas:
        estado = "REVISAR"
    fila["ESTADO"] = estado
    fila["FECHA_EXTRACCION"] = date.today().strftime("%d/%m/%Y")
    return df.rename(columns=COLUMNAS), fila, None

# ── 5. BUCLE: CORTE x REGIÓN (con reanudación) ───────────────────────────────
sesion.get(PAGINA, timeout=60)                 # cookies del sitio, como el navegador
hechos = set()
paises_vistos, encontrados = set(), set()         # para avisar si un país no coincide con los datos
if len(control):
    ok = control[(control["ESTADO"] == "OK") & (control["CORTE_COMPLETO"] == "SI")]
    hechos = {(str(r["AÑO"]), r["SEMESTRE"], r["REGION"]) for _, r in ok.iterrows()
              if r.get("ARCHIVO") in datos_drive}

def guardar_control():
    ruta = os.path.join(ruta_guardado, NOMBRE_CONTROL)
    control[COLS_CONTROL].to_csv(ruta, index=False, encoding="utf-8-sig")
    subir(ruta, DRIVE_FOLDER_CONTROL_ID, control_drive)

tareas = [(c, r) for c in CORTES for r in regiones]
pendientes = [(c, r) for c, r in tareas if (str(c[0]), f"S{c[1]}", r) not in hechos]
print(f"Ya descargados con control OK: {len(tareas) - len(pendientes)}  |  pendientes: {len(pendientes)}\n")

errores, resumen = [], []
for n, ((año, sem, d, h, completo), region) in enumerate(pendientes, 1):
    etiqueta = f"[{n}/{len(pendientes)}] {año}-S{sem} {region}"
    df, fila, err = descargar_corte(año, sem, d, h, completo, region)
    if err:
        print(f"❌ {etiqueta}: {err}")
        errores.append({"AÑO": año, "SEMESTRE": f"S{sem}", "REGION": region, "ERROR": err})
        time.sleep(random.uniform(*PAUSA))
        continue
    nombre = nombre_dato(d, h, region)
    ruta = os.path.join(ruta_guardado, nombre)
    df.to_csv(ruta, index=False, encoding="utf-8-sig")
    subio = subir(ruta, DRIVE_FOLDER_ID, datos_drive)
    os.remove(ruta)                                # no llenar el disco de Colab
    if subio:                                      # borrar la versión parcial anterior del semestre
        for viejo in versiones_viejas(año, sem, region, nombre):
            try:
                drive.files().delete(fileId=datos_drive.pop(viejo), supportsAllDrives=True).execute()
                print(f"   🗑️ Reemplazado: {viejo}")
            except Exception as e:
                print(f"   ⚠️ No se pudo borrar {viejo}: {e}")
    fila["ARCHIVO"] = nombre if subio else ""
    clave = (control["AÑO"].astype(str) == str(año)) & (control["SEMESTRE"] == f"S{sem}") & (control["REGION"] == region)
    resto = control[~clave]
    control = pd.DataFrame([fila]) if resto.empty else pd.concat([resto, pd.DataFrame([fila])], ignore_index=True)
    guardar_control()                              # se guarda tras cada corte: permite reanudar
    marca = "✅" if fila["ESTADO"] == "OK" else "⚠️"
    extra = f" → guardadas: {fila['FILAS_GUARDADAS']:,} filas, FOB {fila['FOB_USD_GUARDADO']:,.2f}" if (CAPS or PAISES_SEL) else ""
    print(f"{marca} {etiqueta}: {fila['FILAS_DESCARGADAS']:,} filas | FOB {fila['FOB_USD_DESCARGADO']:,.2f} USD{extra} | {fila['ESTADO']}")
    resumen.append(fila)
    time.sleep(random.uniform(*PAUSA))

# ── 6. RESUMEN ───────────────────────────────────────────────────────────────
print("═" * 70)
print(f"Descargados en esta corrida: {len(resumen)}  |  con error: {len(errores)}")
if resumen:
    display(pd.DataFrame(resumen)["ESTADO"].value_counts().to_frame("cortes"))
    rev = pd.DataFrame(resumen)
    rev = rev[rev["ESTADO"] != "OK"]
    if len(rev):
        print("⚠️ Cortes a revisar (no cuadran con la consulta de la web):")
        display(rev[["AÑO", "SEMESTRE", "REGION", "DIF_FILAS", "DIF_FOB_USD", "REGIONES_EN_DATOS"]])
if errores:
    print("❌ Cortes con error (vuelve a ejecutar el Bloque 3 para reintentarlos):")
    display(pd.DataFrame(errores))
    ruta_log = os.path.join(ruta_guardado, f"LOG Exportaciones {date.today().isoformat()} - {ETIQ_FILTRO}.csv")
    pd.DataFrame(errores).to_csv(ruta_log, index=False, encoding="utf-8-sig")
    subir(ruta_log, DRIVE_FOLDER_CONTROL_ID, control_drive)
if PAISES_SEL and resumen:
    sin_dato = [p for p in PAISES_SEL if norm_txt(p) not in encontrados]
    if sin_dato:
        import difflib
        print(f"ℹ️ Sin filas para: {', '.join(sin_dato)}. Puede que no haya exportaciones, o que el nombre no "
              "coincida con la columna «Mercado destino». Parecidos en los datos:")
        for p in sin_dato:
            print(f"   {p} → {difflib.get_close_matches(p, sorted(paises_vistos), n=3, cutoff=0.5) or '—'}")
faltan = len(pendientes) - len(resumen)
print("✅ Todo descargado." if not faltan else f"ℹ️ Faltan {faltan} corte(s): ejecuta de nuevo el Bloque 3.")
print(f"   Datos:   https://drive.google.com/drive/folders/{DRIVE_FOLDER_ID}")
print(f"   Control: https://drive.google.com/drive/folders/{DRIVE_FOLDER_CONTROL_ID}")